<a href="https://colab.research.google.com/github/psept77-star/CC_success_probability/blob/main/CCSuccessProbabilitywith_large_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score


In [ ]:
# 1. Ingest expanded dataset
df = pd.read_csv('/content/sample_data/change_control_large_dataset.csv')

# 2. Re-engineer features to remove data leakage markers
df['Implementation_success'] = np.where(df['Linked_Deviations_Count'] == 0, 1, 0)

df['change_initiation_date'] = pd.to_datetime(df['change_initiation_date'], dayfirst=True)
df['change_implementation_date'] = pd.to_datetime(df['change_implementation_date'], dayfirst=True)

# Use ONLY features known at the time of prediction execution
df['Days_to_implement'] = (df['change_implementation_date'] - df['change_initiation_date']).dt.days

# Feature Spaces Split
num_features = ['Days_to_implement'] # Removed 'Days_to_close' to eliminate leakage bias
cat_features = ['Department', 'Change_related_to', 'Severity', 'Linked_CAPA_Effectiveness']
text_features = 'Change_Description'

x = df[num_features + cat_features + [text_features]]
y = df['Implementation_success']


In [ ]:
# 3. Stratified Data Splitting
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Expanded Transformer Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
        # Upgraded text feature extraction space for scaled textual context
        ('text', TfidfVectorizer(max_features=1500, stop_words='english', ngram_range=(1,2)), text_features)
    ]
)

In [ ]:
# 5. Model fitting cycle
production_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, random_state=42))
])

production_pipeline.fit(x_train, y_train)

# Evaluation Execution
y_pred = production_pipeline.predict(x_test)
y_prob = production_pipeline.predict_proba(x_test)[:, 1]

print("-------------------- Upgraded Evaluation Report --------------------")
print(classification_report(y_test, y_pred))
print(f"Realistic ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")


-------------------- Upgraded Evaluation Report --------------------
              precision    recall  f1-score   support

           0       0.64      0.71      0.67       480
           1       0.73      0.66      0.69       560

    accuracy                           0.68      1040
   macro avg       0.68      0.69      0.68      1040
weighted avg       0.69      0.68      0.68      1040

Realistic ROC-AUC Score: 0.7553


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create interactive input widgets
dept_widget = widgets.Dropdown(
    options=['Quality Assurance', 'Production', 'IT', 'Regulatory', 'Engineering'],
    value='Quality Assurance',
    description='Department:'
)

relation_widget = widgets.Dropdown(
    options=['Equipment', 'Process', 'Software', 'Documentation', 'Facility'],
    value='Process',
    description='Related To:'
)

severity_widget = widgets.Dropdown(
    options=['Minor', 'Major', 'Critical'],
    value='Minor',
    description='Severity:'
)

capa_widget = widgets.Dropdown(
    options=['Effective', 'Ineffective', 'Pending'],
    value='Ineffective',
    description='CAPA Effect:'
)

days_widget = widgets.IntSlider(
    value=30,
    min=1,
    max=365,
    step=1,
    description='Days to Imp:'
)

desc_widget = widgets.Textarea(
    value='Upgrading autoclave heating element for faster cycle times.',
    description='Description:',
    layout=widgets.Layout(width='400px', height='80px')
)

predict_button = widgets.Button(
    description='Predict CC Success',
    button_style='success',
    tooltip='Click to run prediction model',
    icon='check'
)

output_area = widgets.Output()

def on_predict_clicked(b):
    with output_area:
        clear_output()
        try:
            # Gather values from widgets
            input_data = pd.DataFrame([{
                'Days_to_implement': days_widget.value,
                'Department': dept_widget.value,
                'Change_related_to': relation_widget.value,
                'Severity': severity_widget.value,
                'Linked_CAPA_Effectiveness': capa_widget.value,
                'Change_Description': desc_widget.value
            }])

            # Predict
            probability = production_pipeline.predict_proba(input_data)[0, 1]
            prediction = production_pipeline.predict(input_data)[0]

            print("\n=================== PREDICTION RESULT ===================")
            print(f"Predicted Outcome: {'SUCCESS (1)' if prediction == 1 else 'FAILURE (0)'}")
            print(f"Success Probability: {probability:.2%}")
            print("=========================================================")
        except NameError:
            print("Error: Please make sure you run the training cells above first to build the pipeline!")
        except Exception as e:
            print(f"An error occurred: {str(e)}")

predict_button.on_click(on_predict_clicked)

# Display the structured UI dashboard
input_form = widgets.VBox([
    widgets.HTML("<h3>Change Control Success Predictor Dashboard</h3>"),
    dept_widget,
    relation_widget,
    severity_widget,
    capa_widget,
    days_widget,
    desc_widget,
    widgets.Label(""),
    predict_button,
    output_area
])

display(input_form)

### Batch Prediction for Uploaded Data

Upload a CSV file containing your change control data to predict the success probability for all records at once and identify the highest probability changes. The CSV should contain columns such as: `Days_to_implement`, `Department`, `Change_related_to`, `Severity`, `Linked_CAPA_Effectiveness`, and `Change_Description`.

In [ ]:
import io
import ipywidgets as widgets
from IPython.display import display, clear_output

upload_widget = widgets.FileUpload(
    accept='.csv',
    multiple=False,
    description='Upload CSV'
)

batch_output_area = widgets.Output()

def on_upload_change(change):
    with batch_output_area:
        clear_output()
        if not upload_widget.value:
            return

        try:
            # Retrieve the uploaded file content
            uploaded_file = list(upload_widget.value.values())[0]
            content = uploaded_file['content']

            # Load into DataFrame
            uploaded_df = pd.read_csv(io.BytesIO(content))

            # Validate required columns
            required_cols = ['Days_to_implement', 'Department', 'Change_related_to', 'Severity', 'Linked_CAPA_Effectiveness', 'Change_Description']
            missing_cols = [col for col in required_cols if col not in uploaded_df.columns]

            if missing_cols:
                print(f"Error: The uploaded CSV is missing the following required columns: {missing_cols}")
                return

            # Generate predictions using the trained production pipeline
            features_df = uploaded_df[required_cols]
            probabilities = production_pipeline.predict_proba(features_df)[:, 1]
            predictions = production_pipeline.predict(features_df)

            # Append results to the dataframe
            uploaded_df['Predicted_Outcome'] = ['Success' if pred == 1 else 'Failure' for pred in predictions]
            uploaded_df['Success_Probability'] = probabilities

            # Sort by success probability descending
            result_df = uploaded_df.sort_values(by='Success_Probability', ascending=False)

            print(f"\nSuccessfully processed {len(result_df)} records!")
            print("Here are the top change controls ranked by success probability:")
            display(result_df[['Days_to_implement', 'Department', 'Change_related_to', 'Severity', 'Success_Probability', 'Predicted_Outcome']].head(20))

            # Option to download the predictions
            csv_buffer = io.StringIO()
            result_df.to_csv(csv_buffer, index=False)
            print("\n(Optional) Copy the processed data or use the saved table above.")

        except Exception as e:
            print(f"An error occurred while processing the file: {str(e)}")

upload_widget.observe(on_upload_change, names='value')

batch_dashboard = widgets.VBox([
    widgets.HTML("<h4>Batch Prediction Dashboard</h4>"),
    upload_widget,
    batch_output_area
])

display(batch_dashboard)